# Аудио-пайплайн MVP
Транскрипция · Техническое качество · Флаг Унылость (в разработке)

## 1. Установка yt-dlp с Deno (для обхода ограничений YouTube)

In [ ]:
import os

# Устанавливаем Deno (нужен yt-dlp для некоторых видео)
!curl -fsSL https://deno.land/install.sh | sh
os.environ["PATH"] += ":/root/.deno/bin"

# Устанавливаем yt-dlp с поддержкой JS-экстракторов
!pip install -U "yt-dlp[default]" -q

# Проверяем
!yt-dlp --version

## 2. Клонируем репозиторий

In [ ]:
# Меняй на свой репозиторий
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

!git clone {REPO_URL}

# Определяем название папки из URL
repo_name = REPO_URL.split("/")[-1].replace(".git", "")
%cd {repo_name}

## 3. Устанавливаем зависимости

In [ ]:
!pip install -r requirements.txt -q

# silero-VAD устанавливается через torch.hub автоматически при первом вызове
# но можно прогреть кэш заранее
import torch
torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    trust_repo=True,
)
print("Зависимости установлены")

## 4. Проверяем GPU

In [ ]:
import torch
from config import DEVICE, COMPUTE_TYPE

print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Пайплайн использует: device={DEVICE}, compute_type={COMPUTE_TYPE}")

## 5. Запуск пайплайна

In [ ]:
import json
import sys
sys.path.insert(0, 'pipeline')  # добавляем папку pipeline в path

from pipeline import run

# Вставь ссылку на тестовое видео
URL = "https://www.youtube.com/watch?v=XXXXX"

result = run(URL)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 6. Просмотр отдельных метрик

In [ ]:
# Транскрипция
print(f"Язык: {result['transcription']['language']}")
print(f"WPM: {result['transcription']['wpm']}")
print(f"Первые 300 символов: {result['transcription']['text'][:300]}")

In [ ]:
# Качество звука
aq = result['audio_quality']
print(f"DNSMOS ovrl: {aq['dnsmos']['ovrl_mos']} ({aq['dnsmos']['quality']})")
print(f"DNSMOS sig:  {aq['dnsmos']['sig_mos']}")
print(f"DNSMOS bak:  {aq['dnsmos']['bak_mos']}")
print(f"LUFS:        {aq['lufs']['value']} dB ({aq['lufs']['quality']})")
print(f"Клиппинг:    {aq['clipping']['detected']} (ratio={aq['clipping']['ratio']})")
print(f"SNR:         {aq['snr']['db']} dB ({aq['snr']['quality']})")

In [ ]:
# Сохранить результат в файл
with open('result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print("Сохранено в result.json")